# Ghostexec + Unsloth + GRPO (Colab / AWS / Kaggle / SageMaker)

Runs **Ghostexec** — an OpenEnv RL environment for executive chief-of-staff decision-making — with **Unsloth + TRL GRPO**. Structure mirrors Unsloth's [OpenEnv 2048 GRPO Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_%2820B%29_Reinforcement_Learning_2048_Game.ipynb), with 2048 / OpenSpiel swapped for the `ghostexec` environment cloned from GitHub (`https://github.com/false200/ghostexec`) and a smaller instruction-tuned model that fits a single T4/A10/A100.

**Default model:** `unsloth/Qwen3-4B-Instruct-2507` (4-bit bnb). This is the **best-in-class small agentic model** as of 2026: on Qwen's published benchmarks it **beats GPT-4.1-nano** on BFCL-v3 tool-calling (61.9 vs 53.0), TAU-Retail agent (48.7 vs 23.5), TAU-Airline agent (32.0 vs 14.0), IFEval (83.4 vs 74.5), and Arena-Hard v2 (43.4 vs 15.9) — while fitting in **~5 GB VRAM** on a free Colab T4. Non-thinking mode means it emits clean JSON without `<think>` blocks, 262K native context, Apache-2.0. Swap via `os.environ['GHOSTEXEC_MODEL_NAME']` to `unsloth/Qwen3-8B` on A10, `unsloth/Qwen2.5-14B-Instruct` on A100, or stay on `unsloth/Qwen3-4B-Instruct-2507` everywhere — it is genuinely that good for structured agentic output.

**What it does:**
1. Clones **OpenEnv** + **ghostexec** from GitHub and installs both.
2. Loads the base model with Unsloth (4-bit) + attaches a LoRA adapter.
3. Demos one `env.reset()` / `env.step()` so you can see the briefing text and the reward breakdown.
4. Builds the GRPO prompt (model must output one `GhostexecAction` JSON object).
5. Runs **GRPOTrainer** with the ghostexec reward channels (`REWARD_FUNCS_ALL`: env-step reward + format validity + group diversity + ID relevance + VIP-critical reply bonus).
6. Runs a before/after generation so you can eyeball the policy change.

**Runtime → Run all** from the top on any GPU runtime. Works in **Colab**, **Kaggle** (enable Internet), **AWS SageMaker / EC2**, **Lambda**, or local.

**Docs:** [Unsloth RL guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) · [OpenEnv](https://github.com/meta-pytorch/OpenEnv) · [Ghostexec](https://github.com/false200/ghostexec)

# Installation

Pinned install stack copied from Unsloth's reference OpenEnv notebook. `uv pip` keeps the resolve fast; `transformers==4.56.2` + `trl==0.22.2` is the combo that imports `GRPOTrainer` cleanly on Colab (newer TRL has mergekit / Pydantic churn).

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

Next we install **OpenEnv** (pinned commit, same as Unsloth's reference notebook) and clone **ghostexec** from GitHub, then `pip install -e` it so `from ghostexec... import ...` just works:

In [ ]:
# 1) Refresh ghostexec to latest main
%cd /content
!rm -rf ghostexec
!git clone https://github.com/false200/ghostexec.git
%cd /content/ghostexec
!git log -1 --oneline

# 2) Install OpenEnv + ghostexec cleanly
%cd /content
!rm -rf OpenEnv
!git clone https://github.com/meta-pytorch/OpenEnv.git
%cd /content/OpenEnv
!git checkout 83dda10
!pip install -q -e .

%cd /content/ghostexec
!pip install -q -e .

# ensure imports resolve from cloned repos
import sys
sys.path.insert(0, "/content/OpenEnv")
sys.path.insert(0, "/content/OpenEnv/src")
sys.path.insert(0, "/content/ghostexec")

ghostexec_root = "/content/ghostexec"

Load **`unsloth/Qwen3-4B-Instruct-2507`** — the best-in-class small agentic model for GRPO + structured JSON in 2026. It **outperforms GPT-4.1-nano on every tool-calling and agent benchmark** (BFCL-v3, TAU-Retail, TAU-Airline, IFEval, Arena-Hard v2) while running in ~5 GB VRAM at 4-bit on a free Colab T4. Non-thinking mode → the output is just the JSON (no `<think>` block to strip). Override via `GHOSTEXEC_MODEL_NAME` env var if you want `Qwen3-8B` (A10+) or `Qwen2.5-14B-Instruct` / `gpt-oss-20b` (A100).

In [ ]:
import os
from unsloth import FastLanguageModel
import torch

MODEL_NAME = os.environ.get("GHOSTEXEC_MODEL_NAME", "unsloth/Qwen3-4B-Instruct-2507")
max_seq_length = 2048   # briefing + JSON action fit comfortably
lora_rank = 16          # Larger rank = smarter, but slower; 8-32 is the sweet spot for GRPO

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    load_in_4bit = True,
    max_seq_length = max_seq_length,
    offload_embedding = False,
)
print(f"Loaded {MODEL_NAME}  |  max_seq_length={max_seq_length}  |  lora_rank={lora_rank}")

Attach a **LoRA** adapter (~1-5% extra weights) so GRPO only updates those. Target the standard Qwen/Llama projection set — works across all the swap-in models above:

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Ghostexec environment with OpenEnv

We use the `GhostexecEnvironment` **in-process** (same object GRPO's reward funcs use). Each call is a fresh episode on a scenario JSON, so there is no state leakage between rollouts. Available scenarios: `phase2_core.json` (default, dense inbox + calendar), `monday_morning.json` (stacked conflicts), `dinner_disaster.json` (personal/professional collision).

In [ ]:
from pathlib import Path
from ghostexec.models import GhostexecAction
from ghostexec.server.ghostexec_environment import GhostexecEnvironment

SCENARIO_PATH = Path(ghostexec_root) / "scenarios" / "phase2_core.json"
os.environ.setdefault("GHOSTEXEC_GRPO_SCENARIO", str(SCENARIO_PATH))

env = GhostexecEnvironment(SCENARIO_PATH)
obs = env.reset()
print(f"Briefing bytes: {obs.message_length}")
print(f"Initial reward: {obs.reward}  done={obs.done}")
print("--- metadata keys:", sorted((obs.metadata or {}).keys()))

Here is the plain-text **briefing** the LLM receives on every turn — inbox, calendar, tasks, active conflicts. This is what GRPO's policy model sees in the user message:

In [ ]:
print(obs.echoed_message[:2000])
print("... (truncated)" if len(obs.echoed_message) > 2000 else "")

Legal actions (from `GhostexecActionType`): `reply_email`, `archive_email`, `reschedule_meeting`, `cancel_meeting`, `complete_task`, `delegate_task`, `send_message`, `do_nothing`. Let's take a **good action** — reply to a critical email — and inspect the reward breakdown:

In [ ]:
# Grab the first critical unreplied email id from the briefing metadata
world = obs.metadata.get("world", {}) if obs.metadata else {}
critical_emails = [e for e in world.get("emails", []) if e.get("priority") == "critical" and not e.get("replied")]
target_email = critical_emails[0] if critical_emails else world.get("emails", [{}])[0]
email_id = target_email.get("id", "e01")
print(f"Replying to email id={email_id} priority={target_email.get('priority')} sender_rel={target_email.get('sender_relationship')}")

good_action = GhostexecAction(
    action_type = "reply_email",
    email_id = email_id,
    message_body = "Acknowledged — I am prioritizing this now and will send revised numbers within the hour.",
)
obs2 = env.step(good_action)
print(f"Reward: {obs2.reward:+.4f}   done={obs2.done}")
print("Breakdown:", obs2.metadata.get("reward_breakdown"))

Now a **do_nothing** — the env always applies a strict additive idle penalty (`-0.15`) on top of whatever the sub-scores produced, so the policy cannot collect zero-cost free rewards by stalling:

In [ ]:
env.reset()
obs_idle = env.step(GhostexecAction(action_type="do_nothing"))
print(f"do_nothing reward: {obs_idle.reward:+.4f}")
print("Breakdown:", obs_idle.metadata.get("reward_breakdown"))

And an **invalid action** — referencing a non-existent email. The env does NOT crash; it returns a structured `step_error` in metadata and the reward picks up a `-0.25` invalid-step adjustment (no idle floor on top, so a bad action is still strictly better than spamming `do_nothing` when the sub-scores happen to be positive):

In [ ]:
env.reset()
bad_action = GhostexecAction(action_type="reply_email", email_id="NOT_AN_EMAIL_ID", message_body="oops")
obs_bad = env.step(bad_action)
print(f"invalid reward: {obs_bad.reward:+.4f}   step_ok={obs_bad.metadata.get('step_ok')}   err={obs_bad.metadata.get('step_error')}")
print("Breakdown:", obs_bad.metadata.get("reward_breakdown"))

# RL task setup

Apply GRPO training defaults (multi-turn rollouts, mild tanh squash, tie-break jitter for zero-variance groups) and build the prompt the policy will see on every GRPO sample. The model must reply with **exactly one JSON object** matching `GhostexecAction` — no markdown fences.

In [ ]:
from ghostexec.notebook_setup import apply_grpo_training_defaults
apply_grpo_training_defaults()
for k in sorted(k for k in os.environ if k.startswith("GHOSTEXEC_") or k == "TRL_EXPERIMENTAL_SILENCE"):
    print(f"  {k}={os.environ[k]}")

In [ ]:
SYSTEM_PROMPT = """You are Ghostexec, an executive chief-of-staff agent.

You receive a plain-text briefing (inbox, calendar, contacts, tasks). You MUST respond with
EXACTLY one JSON object (no markdown fences, no commentary) matching GhostexecAction:
- action_type (required string): one of
  reply_email, archive_email, reschedule_meeting, cancel_meeting, complete_task,
  delegate_task, send_message, do_nothing
- Optional fields as needed: email_id, message_body, meeting_id, new_time, reason,
  task_id, contact_name, message

Prefer high-impact, legal actions on unread critical / VIP items. Never return do_nothing unless the inbox, calendar, and tasks are fully clean."""

# Fresh env → briefing text for the user prompt (GRPO will reuse a fresh env per completion inside the reward func)
env.reset()
briefing_obs = env.reset()
BRIEFING_TEXT = briefing_obs.echoed_message

USER_PROMPT = f"""Read the briefing and output one JSON object for the best immediate executive action.

=== BRIEFING ===
{BRIEFING_TEXT}"""

print(USER_PROMPT[:1500])
print("... (truncated)" if len(USER_PROMPT) > 1500 else "")

First, let's prompt the **untrained** model and see what it emits without RL:

In [ ]:
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# Reward functions

We reuse `REWARD_FUNCS_ALL` from `training/grpo_ghostexec_reward.py` — already battle-tested with 133 in-process pytest cases and a 500+ live-API dead-suite. GRPO will **average the reward channels per completion**:

- **`ghostexec_env_step_reward`** — spins up a fresh `GhostexecEnvironment` per completion, parses the JSON into a `GhostexecAction`, and steps the env. This is the dense potential-based reward: conflict / relationship / task sub-scores (weights 0.35 / 0.35 / 0.30), tanh-squashed into (-1, 1) with a strict `-0.15` idle floor and `-0.25` invalid-step adjustment.
- **`reward_format_valid`** — `+1` if the completion parses into a legal `GhostexecAction`, `-1` if not.
- **`reward_group_diversity`** — penalizes duplicate completions inside a GRPO group (prevents collapse).
- **`reward_id_relevance`** — small penalty when the action cites an email / task / meeting ID that does not exist in the scenario.
- **`reward_vip_critical_reply_bonus`** — `+0.5` when the action is a non-empty reply to an unreplied VIP-critical email.

In [ ]:
from ghostexec.training.grpo_ghostexec_reward import REWARD_FUNCS_ALL
for fn in REWARD_FUNCS_ALL:
    print(f"  - {fn.__name__}")

We'll now create a dataset where every row is a copy of the same prompt (GRPO draws `num_generations` completions per prompt and computes advantages inside the group). The env itself provides the environment-stochasticity — each reward-func call resets a fresh `GhostexecEnvironment`. `GHOSTEXEC_PERTURB=1` + `GHOSTEXEC_CURRICULUM=all` will rotate / shuffle scenarios if you want more diversity.

In [ ]:
from datasets import Dataset

dataset_size = int(os.environ.get("GHOSTEXEC_DATASET_SIZE", "1000"))
row = {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    "answer": 0,
}
dataset = Dataset.from_list([row] * dataset_size)

maximum_length = len(
    tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True)
)
print(f"dataset size: {len(dataset)}   tokenized prompt length: {maximum_length}")

<a name="Train"></a>
# GRPO config + trainer

Same knobs as Unsloth's reference OpenEnv notebook, tuned for **Qwen3-4B on a single T4**. The 4B base + LoRA + reference model + 4 generations fits comfortably in 16 GB. On A10/A100 bump `num_generations` to 6-8 and `max_steps` up; on a 12 GB GPU drop `num_generations` to 2 or `max_completion_length` to 384.

In [ ]:
max_prompt_length = maximum_length + 16
max_completion_length = max_seq_length - max_prompt_length

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = int(os.environ.get("GHOSTEXEC_MAX_STEPS", "300")),
    save_steps = 100,
    report_to = "trackio",
    output_dir = "outputs",
    seed = 3407,
)
print(f"max_prompt_length={max_prompt_length}  max_completion_length={max_completion_length}  max_steps={training_args.max_steps}")

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = REWARD_FUNCS_ALL,
    args = training_args,
    train_dataset = dataset,
)

And let's run the trainer. Watch the `reward` column — it should trend up. A per-step log also lands under `outputs/` (TrackIO). **Note:** GRPO is slow; 300 steps on a T4 with Qwen3-4B + 4 generations ≈ 40-60 min. Shrink `GHOSTEXEC_MAX_STEPS` for a quick smoke run.

In [ ]:
trainer.train()

<a name="Inference"></a>
# After-training inference

Sample from the same prompt so you can eyeball the policy shift — expect more `reply_email` / `reschedule_meeting` / `complete_task` with real IDs and less `do_nothing`.

In [ ]:
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
# Save / push

Flip the `if False:` guards to save locally or push to the Hugging Face Hub.

In [ ]:
# Merge + save locally in 16-bit
if False:
    model.save_pretrained_merged("ghostexec_finetuned", tokenizer, save_method = "merged_16bit")

# Push LoRA-only to HF Hub
if False:
    model.push_to_hub("hf-user/ghostexec-grpo-lora", token = "hf_xxx")

# Push merged 16-bit to HF Hub
if False:
    model.push_to_hub_merged("hf-user/ghostexec-grpo-merged", tokenizer, save_method = "merged_16bit", token = "hf_xxx")

# Done

You now have a GRPO-trained LoRA on top of your base model, optimised against Ghostexec's conflict / relationship / task reward blend with format + diversity + ID + VIP-critical side channels. Check `outputs/` for TrackIO logs and checkpoints, and the Ghostexec repo's `scripts/generate_committed_submission_plots.py` for the reward-curve + per-channel plots that go into the hackathon submission.